# Live: Earthquake Activity (last hour)

Polls the USGS "past hour" earthquake feed (updated by USGS every ~1
minute) on a fixed interval and re-renders an interactive Plotly map each
time, so the notebook visibly refreshes with new data while it runs — a
genuinely real-time demo, not a canned animation.

Pure Python + pandas + Plotly — no Spark session needed, since this reads
directly from the live feed rather than a persisted table. For a durable,
queryable copy of earthquake data see
`data-generators/bronze_ingest_usgs_earthquakes.ipynb`.

Requires `plotly` and `pandas` (see `requirements.txt` in this folder).

In [ ]:
import json
import time
import urllib.request
from datetime import datetime, timezone

import pandas as pd
import plotly.express as px
from IPython.display import clear_output

## Configuration

`N_REFRESHES` bounds the loop so the demo doesn't run forever — bump it up
for a longer-running live dashboard, or re-run the last cell any time for
a single fresh snapshot.

In [ ]:
SOURCE_URL = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_hour.geojson"
REFRESH_SECONDS = 60
N_REFRESHES = 5

## Fetch + flatten one snapshot

In [ ]:
def fetch_quakes_df():
    with urllib.request.urlopen(SOURCE_URL) as resp:
        data = json.load(resp)

    rows = []
    for feature in data["features"]:
        props = feature["properties"]
        lon, lat, depth = feature["geometry"]["coordinates"][:3]
        rows.append({
            "id": feature["id"],
            "magnitude": props.get("mag"),
            "place": props.get("place"),
            "longitude": lon,
            "latitude": lat,
            "depth_km": depth,
            "event_time": pd.to_datetime(props.get("time"), unit="ms"),
        })

    df = pd.DataFrame(rows)
    return df[df["magnitude"].notna()].copy()

## Live loop

Each iteration fetches a fresh snapshot, clears the previous plot, and
renders a new one — magnitude drives both marker color and size, hover
shows place/depth.

In [ ]:
for i in range(N_REFRESHES):
    quakes_df = fetch_quakes_df()
    quakes_df["marker_size"] = quakes_df["magnitude"].abs().clip(lower=0.5)
    refreshed_at = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

    fig = px.scatter_geo(
        quakes_df,
        lat="latitude",
        lon="longitude",
        color="magnitude",
        size="marker_size",
        hover_name="place",
        hover_data={"magnitude": True, "depth_km": True, "marker_size": False},
        color_continuous_scale="Inferno",
        projection="natural earth",
        title=f"Live Earthquake Activity — last hour ({len(quakes_df)} events) — refreshed {refreshed_at}",
    )
    fig.update_layout(
        template="plotly_dark",
        height=560,
        margin=dict(l=0, r=0, t=60, b=0),
        geo=dict(bgcolor="rgba(0,0,0,0)", showland=True, landcolor="#1f2430"),
    )

    clear_output(wait=True)
    fig.show()
    print(f"Refresh {i + 1}/{N_REFRESHES} — {len(quakes_df)} events in the last hour")

    if i < N_REFRESHES - 1:
        time.sleep(REFRESH_SECONDS)